In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def standardize_weather_columns(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("region", F.upper(F.trim(F.col("region"))))
        .withColumn("weather_alert_level", F.upper(F.trim(F.col("weather_alert_level"))))
    )

def cast_weather_fields(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("temperature_c", F.col("temperature_c").cast("double"))
        .withColumn("wind_speed_kmh", F.col("wind_speed_kmh").cast("double"))
        .withColumn("precipitation_mm", F.col("precipitation_mm").cast("double"))
    )

def add_weather_day(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("report_day", F.to_date("event_date"))
    )

def filter_invalid_weather(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(F.col("temperature_c").isNotNull())
        .filter(F.col("wind_speed_kmh").isNotNull())
        .filter(F.col("report_day").isNotNull())
    )


def transform_weather(df: DataFrame) -> DataFrame:
    """Complete weather transformation pipeline"""
    return (
        df
        .transform(standardize_weather_columns)
        .transform(cast_weather_fields)
        .transform(add_weather_day)
        .transform(filter_invalid_weather)
    )

In [0]:
# Load configuration
import yaml

config_path = "/Workspace/Repos/adb-emily@startsteps.org/vattenfall-week9-capstone-EmilyImunde/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
bronze_schema = config["schemas"]["raw"]
silver_schema = config["schemas"]["refined"]

print(f"Catalog: {catalog_name}")
print(f"Bronze Schema: {bronze_schema}")
print(f"Silver Schema: {silver_schema}")

In [0]:
bronze_df = spark.table(f"{catalog_name}.{bronze_schema}.bronze_weather")
silver_df = transform_weather(bronze_df)
silver_df.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{silver_schema}.silver_weather"
)

print(f"✓ Successfully created {catalog_name}.{silver_schema}.silver_weather")
print(f"Total rows: {silver_df.count()}")

In [0]:

silver_table = spark.table("vattenfall_dev.refined.silver_weather")

print(f"Total rows: {silver_table.count()}")
print("\nSchema:")
silver_table.printSchema()
print("\nFirst 10 rows:")
display(silver_table.limit(10))

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM vattenfall_dev.refined.silver_weather;